In [ ]:
import sys
#CHANGED APPROACH TO WORK WITH MASSES INSTEAD OF AA to get lower runtime
# Please do not remove package declarations because these are used by the autograder.

MASSES = {
    "G": 57, "A": 71, "S": 87, "P": 97, "V": 99, "T": 101, "C": 103, "I": 113, "L": 113,
    "N": 114, "D": 115, "K": 128, "Q": 128, "E": 129, "M": 131, "H": 137, "F": 147,
    "R": 156, "Y": 163, "W": 186,
}

# Create a list of unique amino acid masses (this is the changed part)
UNIQUE_MASSES = sorted(set(MASSES.values()))

def Mass(peptide):
    return sum(peptide)

def Expand(Leaderboard):
    newLB = set()
    for pep in Leaderboard:
        for mass in UNIQUE_MASSES:
            newLB.add(pep + (mass,))
    return newLB

def LinearSpectrum(peptide):
    preMass = [0]
    for mass in peptide:
        preMass.append(preMass[-1] + mass)
    
    linear_spectrum = [0]
    for i in range(len(peptide)):
        for j in range(i + 1, len(peptide) + 1):
            
            linear_spectrum.append(preMass[j] - preMass[i])
    return sorted(linear_spectrum)

def CyclicSpectrum(peptide):
    n = len(peptide)
    preMass = [0]
    for mass in peptide:
        preMass.append(preMass[-1] + mass)
    
    cyclic_spectrum = [0]
    for i in range(n):
        for j in range(1, n):
            if i + j <= n:
                cyclic_spectrum.append(preMass[i + j] - preMass[i])
            else:
                cyclic_spectrum.append(preMass[n] - (preMass[i] - preMass[i + j - n]))
    
    cyclic_spectrum.append(preMass[n])  # full mass
    
    return sorted(cyclic_spectrum)

def LinearScore(pep_spec, spectrum):
    score = 0
    spectrum_copy = spectrum[:]
    for value in pep_spec:
        if value in spectrum_copy:
            score += 1
            spectrum_copy.remove(value)
    return score

def cyclopeptide_scoring(peptide, spectrum):
    cyclic_spec = CyclicSpectrum(peptide)
    return LinearScore(cyclic_spec, spectrum)

def Trim(Leaderboard, spectrum, n):
    if not Leaderboard:
        return set()
    
    scores = []
    for pep in Leaderboard:
        pep_spec = LinearSpectrum(pep)
        score = LinearScore(pep_spec, spectrum)
        scores.append((score, pep))
    scores.sort(reverse=True) 
    
    if len(scores) <= n:
        return set([pep for score,pep in scores])
    
    # to get the scores in order but I need to now take the ties also when there are more than n peps
    nth_score = scores[n - 1][0]
    return set([pep for score, pep in scores if score >= nth_score])

def leaderboard_cyclopeptide_sequencing(spectrum: list[int], n: int) -> list[int]:
    
    Leaderboard = {()}  
    LeaderPeptide = ()
    LeaderScore = 0
    parentmass = spectrum[-1]
    
    while Leaderboard:
        Leaderboard = Expand(Leaderboard)
        
        pep_to_remove = set()
        
        for Peptide in Leaderboard:
            peptide_mass = Mass(Peptide)
            # Complete peptide 
            if peptide_mass == parentmass:
                
                pep_to_remove.add(Peptide)
                PeptideScore = cyclopeptide_scoring(Peptide, spectrum)
                if PeptideScore > LeaderScore:
                    LeaderPeptide = Peptide
                    LeaderScore = PeptideScore

            #if it is larger (dont want to expand anymore)
            elif peptide_mass > parentmass:
                pep_to_remove.add(Peptide)
        
        # using pep to remove to avoid mutations like in the previous case
        Leaderboard -= pep_to_remove
        
        # Trim to top n and expand (restart while loop)
        Leaderboard = Trim(Leaderboard, spectrum, n)
    
    return list(LeaderPeptide)

In [ ]:
# Code Challenge: ImplementLeaderboardCyclopeptideSequencing().

# Input: An integer N and a collection of integers Spectrum.
# Output: LeaderPeptide after running LeaderboardCyclopeptideSequencing(Spectrum, N).
# Note: Multiple solutions may exist. You may return any one.

# Sample Input:

# 0 71 113 129 147 200 218 260 313 331 347 389 460
# 10
# Sample Output:

# 71-147-113-129